# Phase 12: Debugging & Best Practices

## What you'll learn
- Model summary with torchinfo
- Anomaly detection
- Reproducibility (seeds, deterministic mode)
- Memory management
- Common pitfalls and how to avoid them
- Unit testing models

---

In [ ]:
import torch
import torch.nn as nn
import random
import numpy as np

## 12.1 Model Summary with torchinfo

See layer-by-layer shapes, parameter counts, and memory usage.

In [ ]:
# pip install torchinfo
from torchinfo import summary

model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
    nn.Linear(128, 10)
)

summary(model, input_size=(1, 3, 32, 32), col_names=['input_size', 'output_size', 'num_params'])

## 12.2 Anomaly Detection

Detect NaN/Inf gradients during training — helps find the exact operation that causes issues.

In [ ]:
# Enable anomaly detection (slower, use only for debugging)
torch.autograd.set_detect_anomaly(True)

x = torch.tensor(1.0, requires_grad=True)
y = x ** 2
y.backward()
print(f"Normal gradient: {x.grad}")

# This would raise an error with anomaly detection:
# x = torch.tensor(-1.0, requires_grad=True)
# y = torch.sqrt(x)  # sqrt of negative → NaN
# y.backward()  # anomaly detection catches this!

# Disable when done debugging
torch.autograd.set_detect_anomaly(False)
print("Anomaly detection demo complete")

## 12.3 Reproducibility

Set all random seeds for reproducible results.

In [ ]:
def set_seed(seed=42):
    """Set all seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic algorithms (may be slower)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

# Verify reproducibility
a = torch.randn(3)
set_seed(42)
b = torch.randn(3)
print(f"Reproducible: {torch.equal(a, b)}")  # True

## 12.4 Memory Management

In [ ]:
if torch.cuda.is_available():
    # Check memory usage
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
    print(f"Cached:    {torch.cuda.memory_reserved() / 1e6:.1f} MB")

    # Create large tensor
    big = torch.randn(10000, 10000, device='cuda')
    print(f"\nAfter allocation: {torch.cuda.memory_allocated() / 1e6:.1f} MB")

    # Delete and clear cache
    del big
    torch.cuda.empty_cache()
    print(f"After cleanup: {torch.cuda.memory_allocated() / 1e6:.1f} MB")

    # Memory snapshot for debugging leaks
    print(f"\nMemory summary:")
    print(torch.cuda.memory_summary(abbreviated=True))
else:
    print("GPU not available — memory management applies to CUDA tensors")

## 12.5 Common Pitfalls

Mistakes that every PyTorch beginner makes (and how to fix them).

In [ ]:
# ❌ PITFALL 1: Forgetting to zero gradients
x = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"  Step {i}: grad = {x.grad}")  # accumulates: 4, 8, 12

# ✅ FIX: Always zero gradients
print("\nWith zeroing:")
x = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    if x.grad is not None:
        x.grad.zero_()
    y = x ** 2
    y.backward()
    print(f"  Step {i}: grad = {x.grad}")  # always 4

In [ ]:
# ❌ PITFALL 2: Forgetting model.eval() during inference
model = nn.Sequential(nn.Linear(10, 5), nn.Dropout(0.5), nn.Linear(5, 1))

x = torch.ones(1, 10)
model.train()
results_train = [model(x).item() for _ in range(5)]
print(f"Train mode (inconsistent): {results_train}")

# ✅ FIX
model.eval()
with torch.no_grad():
    results_eval = [model(x).item() for _ in range(5)]
print(f"Eval mode (consistent):    {results_eval}")

In [ ]:
# ❌ PITFALL 3: Tensors on different devices
a = torch.tensor([1.0])  # CPU
# b = torch.tensor([2.0], device='cuda')  # GPU
# c = a + b  # RuntimeError!

# ✅ FIX: Move to same device
# a = a.to('cuda')
# c = a + b  # works
print("Always ensure tensors are on the same device")

In [ ]:
# ❌ PITFALL 4: Using softmax with CrossEntropyLoss
# CrossEntropyLoss already applies LogSoftmax internally!

logits = torch.randn(4, 10)
labels = torch.randint(0, 10, (4,))
criterion = nn.CrossEntropyLoss()

# ❌ WRONG: double softmax
# loss = criterion(torch.softmax(logits, dim=1), labels)

# ✅ CORRECT: pass raw logits
loss = criterion(logits, labels)
print(f"Correct loss: {loss.item():.4f}")

## 12.6 Unit Testing Models

Quick sanity checks before training.

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

def test_model():
    model = MyModel()

    # Test 1: Output shape
    x = torch.randn(8, 784)
    out = model(x)
    assert out.shape == (8, 10), f"Expected (8,10), got {out.shape}"
    print("✅ Output shape correct")

    # Test 2: No NaN in output
    assert not torch.isnan(out).any(), "NaN in output!"
    print("✅ No NaN in output")

    # Test 3: Can overfit a single batch
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    x = torch.randn(4, 784)
    y = torch.tensor([0, 1, 2, 3])

    for _ in range(100):
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    assert loss.item() < 0.01, f"Can't overfit single batch: loss={loss.item()}"
    print(f"✅ Overfits single batch (loss={loss.item():.6f})")

    # Test 4: Parameters update after backward
    model2 = MyModel()
    params_before = [p.clone() for p in model2.parameters()]
    opt = torch.optim.SGD(model2.parameters(), lr=0.1)
    opt.zero_grad()
    criterion(model2(x), y).backward()
    opt.step()
    params_after = list(model2.parameters())

    changed = any(not torch.equal(b, a) for b, a in zip(params_before, params_after))
    assert changed, "Parameters didn't update!"
    print("✅ Parameters update after step")

test_model()

## ✅ Phase 12 Summary

You now know how to:
- Inspect models with torchinfo
- Detect gradient anomalies (NaN/Inf)
- Ensure reproducibility with seeds
- Manage GPU memory
- Avoid the most common PyTorch pitfalls
- Write sanity-check tests for models

**Next → Phase 13: Deployment**